In [1]:
import json
from datasets import load_dataset

from evalforge.utils import pprint

## Load the dataset

In [2]:
# https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

DATASET_NAME = "Amazon-Reviews-2023"
CATEGORY = "Clothing_Shoes_and_Jewelry" # beware, this is huge!


In [3]:
category = CATEGORY

# def load_category(category):
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                    f"raw_review_{category}", split="full", trust_remote_code=True)
dataset_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                        f"raw_meta_{category}", split="full", trust_remote_code=True)
print(f"Loaded {len(dataset)} reviews and {len(dataset_meta)} metadata")
pprint(dataset[0])
print("-"*100)
pprint(dataset_meta[0])
# return dataset, dataset_meta

Loaded 66033346 reviews and 7218481 metadata
{
    "rating": 3.0,
    "title": "Arrived Damaged : liquid in hub locker!",
    "text": "Unfortunately Amazon in their wisdom (cough, cough) decided to ship the snowsuit in a vinyl bag with holes in it!  There was no other bag to protect the snowsuit inside vinyl bag with all the holes.  This is what happened:  Arrived in hub locker. It was the very top locker. Opened it & pulled the pkg out getting a very wet & nasty surprise at the same time. My senses were assaulted. Smells like tea tree oil. Feels like conditioner or lotion.  I can\u2019t understand how the delivery person a) didn\u2019t smell that mess when they shoved the pkg in b) didn\u2019t see the mess when they shoved it in - tho if they were short I guess that would explain it bc I\u2019m 5\u201910\u201d & I didn\u2019t see it until the pkg was in my hands. The locker was up high & dark, but I could smell it the minute I walked into the hub locker room. I happen to be extremely 

In [4]:
pprint(dataset_meta[0])

{
    "main_category": "AMAZON FASHION",
    "title": "BALEAF Women's Long Sleeve Zip Beach Coverup UPF 50+ Sun Protection Hooded Cover Up Shirt Dress with Pockets",
    "average_rating": 4.2,
    "rating_number": 422,
    "features": [
        "90% Polyester, 10% Spandex",
        "Zipper closure",
        "Machine Wash",
        "Long sleeve sun protection coverups--UPF 50+ blocks the sun from burning",
        "Zipped v-neckline--fashionable V neck and smooth 1/4 zipper allows to staying place as you like",
        "Two drop-in side pockets--hold your phone or keys well\uff0cno worries of falling out",
        "Hoodie with non-slip drawcord--Enhancing hooded design is convenient to wrap your face and enough space to put your head and hair easily",
        "A flattering coverups company you spend all day on the beach\uff0ctraveling with lovers or busying around house. Recommended For everyday leisure or daily exercise"
    ],
    "description": [],
    "price": "31.99",
    "images":

## Merge on items and reviews

- `parent_asin` is the ASIN of the product
- `title_meta` is the title of the product
- `title_review` is the title of the review

We are going to sample from the metadata dataset as it contains the actual products data. Once we sample here, we can merge on the reviews dataset.

In [79]:
SAMPLE_SIZE = 10_000

def filter_meta(x):
    cond = (x["parent_asin"] is not None and
            x["main_category"] == "AMAZON FASHION" and
            x["title"] is not None and 
            x["description"] is not None and
            x["average_rating"] > 3.0 and
            x["rating_number"] > 10)
    return cond

dataset_meta_filtered = dataset_meta.filter(filter_meta, num_proc=16)

dataset_meta_sample = dataset_meta_filtered.shuffle(seed=42).select(range(SAMPLE_SIZE))

Filter (num_proc=16): 100%|██████████| 7218481/7218481 [00:32<00:00, 219266.19 examples/s]


In [80]:
ids = dataset_meta_sample["parent_asin"]

In [81]:
#let's filter dataset on parent_asin in dataset_meta_sample
dataset_filtered = dataset.filter(lambda x: x["parent_asin"] in ids, num_proc=16)
dataset_filtered


Filter (num_proc=16): 100%|██████████| 66033346/66033346 [07:21<00:00, 149660.62 examples/s]


Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 159447
})

## Let's merge!

In [82]:
import pandas as pd

# Convert reviews and metadata datasets to pandas DataFrames
metadata_df = pd.DataFrame(dataset_meta_sample)
reviews_df = pd.DataFrame(dataset_filtered)

print(f"Before filtering: {len(metadata_df)} metadata, {len(reviews_df)} reviews")

Before filtering: 10000 metadata, 159447 reviews


In [83]:
# Let's join them on `parent_asin`
merged_df = pd.merge(metadata_df, reviews_df, on="parent_asin", how="inner", suffixes=("_meta", "_review"))
merged_df.head()

,main_category,title_meta,average_rating,rating_number,features,description,price,images_meta,videos,store,...,author,rating,title_review,text,images_review,asin,user_id,timestamp,helpful_vote,verified_purchase
0,AMAZON FASHION,RomenSi Men's Air Cushion Sport Running Shoes ...,4.1,2570,"[Imported, Rubber sole, Air cushion design and...",[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",RomenSi,...,None,1.0,Yikes,The shoes are ugly,[],B089PWT37D,AGX7QVVCXVIPCWSFPW5N3ARAA6DQ,1643372843528,0,True
1,AMAZON FASHION,"SR Max Geneva, Women's, Clog Style Slip Resist...",4.3,1626,"[Rubber sole, Quality full grain leather upper...","[Loved by medical professionals, the soft toe ...",None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SR Max,...,None,5.0,Es lo que esperaba en comodidad,Lo uso 8 horas seguidas y estoy felizzz con el...,[],B005XHJWT0,AGN4VM7KUIFPHYZJF5PRPUSDBEFA,1630537626269,0,True
2,AMAZON FASHION,"SR Max Geneva, Women's, Clog Style Slip Resist...",4.3,1626,"[Rubber sole, Quality full grain leather upper...","[Loved by medical professionals, the soft toe ...",None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SR Max,...,None,5.0,Comfort. I am on my feet for more than 14-15 h...,I like the height and support and everything e...,[],B076KSG5Z6,AF7OFC6JR45BOKA63XCIVHARK2YQ,1646600164868,0,True
3,AMAZON FASHION,"SR Max Geneva, Women's, Clog Style Slip Resist...",4.3,1626,"[Rubber sole, Quality full grain leather upper...","[Loved by medical professionals, the soft toe ...",None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SR Max,...,None,1.0,Sliding forward,These were the worst pair of work shoes I have...,[],B005XHK7BC,AHLXSN2KCUW5LVNQFWSRE4RCMGRA,1641162748032,0,True
4,AMAZON FASHION,"SR Max Geneva, Women's, Clog Style Slip Resist...",4.3,1626,"[Rubber sole, Quality full grain leather upper...","[Loved by medical professionals, the soft toe ...",None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SR Max,...,None,5.0,Pretty good.,One of the shoes was a lot tighter than the ot...,[],B005XHK910,AEY4OGSUEF77W5IMMHYI35QGFKJQ,1551317136173,0,True


In [84]:
len(merged_df)

159447

In [85]:
final_df = merged_df.set_index("parent_asin").sort_index()
final_df.head()

,main_category,title_meta,average_rating,rating_number,features,description,price,images_meta,videos,store,...,author,rating,title_review,text,images_review,asin,user_id,timestamp,helpful_vote,verified_purchase
parent_asin,,,,,,,,,,,,,,,,,,,,,
5781728791,AMAZON FASHION,Women's Crewneck Striped Shirt Loose Colorbloc...,4.0,27,"[Denim, Hand Wash Only, Imported, Features: Cr...",[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Ladiyo,...,None,4.0,Soft and light,Bought for a Waldo costume. A little more burg...,"[{'attachment_type': 'IMAGE', 'large_image_url...",5781728791,AFVXCCSLG6U54JVWKPD2YKZOTPEA,1635625997627,1,True
B0000WL74G,AMAZON FASHION,Carhartt Men's Denim Unlined Bib Overall R08,4.5,1105,"[100% rigid denim, Imported, Machine Wash, Mul...",[Denim bib overall is the perfect workwear cho...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Carhartt,...,None,4.0,"runs super long and wide at the ankles, needed...","Color is good, top and waist fit well. When it...",[],B0000WL74G,AELMD3XERKYVF53RN6Z6I3QWRZUA,1638847561683,1,True
B0000WL74G,AMAZON FASHION,Carhartt Men's Denim Unlined Bib Overall R08,4.5,1105,"[100% rigid denim, Imported, Machine Wash, Mul...",[Denim bib overall is the perfect workwear cho...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Carhartt,...,None,2.0,Size Issues,The top of the overalls almost fit but the leg...,[],B0000WL74G,AGDYXIA7XVX2JZUBWZLOTB7UK67A,1604450417799,0,True
B0000WL74G,AMAZON FASHION,Carhartt Men's Denim Unlined Bib Overall R08,4.5,1105,"[100% rigid denim, Imported, Machine Wash, Mul...",[Denim bib overall is the perfect workwear cho...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Carhartt,...,None,5.0,I didn't expect the elastic straps. I guess I ...,I didn't expect the elastic straps. I guess I ...,[],B0000WL74G,AHWPYBJHUH2QAYFBYNHXEQZJWLYQ,1431728733000,1,True
B0000WL74G,AMAZON FASHION,Carhartt Men's Denim Unlined Bib Overall R08,4.5,1105,"[100% rigid denim, Imported, Machine Wash, Mul...",[Denim bib overall is the perfect workwear cho...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Carhartt,...,None,5.0,Five Stars,My son loved these. Great quality.,[],B0000WL74G,AG4JUDBWJGOSW647FQUCI76MDBJA,1537801397231,0,True


In [86]:
# I want to index on the parent_asin but with an integer index

first_item = final_df.index.get_level_values("parent_asin").unique()[3]
print(first_item)
final_df.loc[first_item]

B0002TOPI2


,main_category,title_meta,average_rating,rating_number,features,description,price,images_meta,videos,store,...,author,rating,title_review,text,images_review,asin,user_id,timestamp,helpful_vote,verified_purchase
parent_asin,,,,,,,,,,,,,,,,,,,,,
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,5.0,Great product,"love gold toe socks, all of them last for year...",[],B0002TOPI2,AEIHFGG7K2HYYUP4BKEUM3FO3VTQ,1565047153660,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,5.0,Gold Toe Woman's Bermuda Turn Cuff Socks are t...,Great socks!,[],B0002TOPI2,AG5ANA6NNZ7PW2EI2FHGGYZQDGGA,1447700777000,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,5.0,Five Stars,Just as expected from the gold toe brand and a...,[],B0002TOPI2,AHCBECDO64PNOKJMHMKFBQL3ZCEA,1454103570000,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,3.0,Three Stars,The knit is a little coarse and not as soft as...,[],B0002TOPI2,AHSQ3H6KAZNC3FOMZKA6JC2GHFPA,1423963861000,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,2.0,Two Stars,These socks were thinner than I expected. Also...,[],B0002TOPI2,AHWBKISCHKZA5LWAZ36ABWOERF3Q,1430389426000,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,3.0,Navy or black?,"I do like these socks, but they are such a dar...",[],B0002TOPI2,AHGNSYDK5BSSJTITLPCOJVANVINQ,1384946717000,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,4.0,Not what I used to order,Wow! Shockingly thin material! I wear lace-up ...,[],B0002TOPI2,AFMNNQGMFSJWLAZ35Y6ZNETL6WCA,1481568164000,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,5.0,Five Stars,"Great sock, great value...",[],B0002TOPI2,AHC2T6SKT4XBXEPHWA2DUR25LU2A,1491832956000,0,True
B0002TOPI2,AMAZON FASHION,"Gold Toe Women's Bermuda Turn Cuff Sock, size ...",4.2,209,"[80% Combed Cotton, 19% Nylon, 1% Spandex, Mac...",[Gold Toe Bermuda turn cuff socks are made of ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",GOLDTOE,...,None,

## Save to disk!

We will flatten the dataframe and save it to disk.

In [87]:
final_df.columns

Index(['main_category', 'title_meta', 'average_rating', 'rating_number',
       'features', 'description', 'price', 'images_meta', 'videos', 'store',
       'categories', 'details', 'bought_together', 'subtitle', 'author',
       'rating', 'title_review', 'text', 'images_review', 'asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')

In [90]:
import numpy as np

# Define which columns we want
METADATA_COLUMNS = [
    "main_category",
    "title_meta",
    "description",
    "average_rating",
    "rating_number",
    "asin",
    "features",
    "price",
    "images_meta",
    "videos",
    "store",
    "categories",
    "details",
    "bought_together",
    "subtitle",
    "author"
]

REVIEW_COLUMNS = [
    "rating",
    "title_review",
    "text",
    "images_review",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]

def create_product_entry(group, metadata_columns=METADATA_COLUMNS, review_columns=REVIEW_COLUMNS):
    # Get metadata from first row
    metadata = {
        "parent_asin": group.index[0],  # Get parent_asin from index
        **{col.replace('_meta', ''): group[col].iloc[0]  # Remove _meta suffix in output
           for col in metadata_columns}
    }
    
    # Create list of reviews
    reviews = group.apply(
        lambda x: {col.replace('_review', ''): x[col]  # Remove _review suffix in output
                  for col in review_columns},
        axis=1
    ).tolist()
    
    return {**metadata, "reviews": reviews}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.int_, np.intc, np.intp, np.int8,
                          np.int16, np.int32, np.int64, np.uint8,
                          np.uint16, np.uint32, np.uint64)):
            return int(obj)
        elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# Process each group and save to JSONL
with open("clothes_review_10k.jsonl", "w") as f:
    for parent_asin, group in final_df.groupby(level=0):
        product_entry = create_product_entry(group)
        f.write(json.dumps(product_entry, cls=NumpyEncoder) + "\n")